# Prepare the candidate dataset

This notebook will read `candidates.parquet` and create the prepared train, validation, and test Parquet files.

## Import dependencies

This block imports only the libraries needed by the preparation notebook. It does not read files or change data.

In [ ]:
from pathlib import Path

import pyarrow.parquet as pq

## Configure the input, output, and date bounds

Set the candidate Parquet location, the directory for the prepared splits, and the optional UTC date window. The `since` bound is inclusive; the `until` bound is exclusive.

In [ ]:
candidates_path = Path("../candidates.parquet")
output_directory = Path("prepared")
since_text = None  # Example: "2024-01-01"
until_text = None  # Example: "2025-01-01"

## Load the candidate Parquet file

In [ ]:
if not candidates_path.is_file():
    raise FileNotFoundError(candidates_path)

candidate_table = pq.read_table(candidates_path)

## Validate the candidate schema

The preparation step expects the six canonical candidate columns. This block checks that they exist, selects them in a stable order, and converts the table into Python rows for the later preparation steps.

In [ ]:
candidate_columns = (
    "id",
    "repository_group_id",
    "subject",
    "diff",
    "committed_at_utc",
    "patch_id",
)
missing_columns = [
    column for column in candidate_columns if column not in candidate_table.column_names
]
if missing_columns:
    raise ValueError(f"Candidate Parquet is missing columns: {', '.join(missing_columns)}")

selected_table = candidate_table.select(list(candidate_columns))
rows = selected_table.to_pylist()
source_count = len(rows)